<a href="https://colab.research.google.com/github/Silva-DTS/Statistical-Learning-e23379/blob/main/Bayesian_Inference_Assignment_Answers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Question 1
###1. Visualizing the Mechanics

The probability that a user answers item \(i\) correctly is given by the **Two-Parameter Logistic (2PL) Item Response Model**

$$
P(Y_i=1|\Theta=\theta)=p_i(\theta)
=\frac{1}{1+e^{-a_i(\theta-b_i)}}.
$$

where

- $\theta$ = user ability
- $a_i$ = discrimination parameter
- $b_i$ = difficulty parameter

The discrimination parameter controls the steepness of the curve, whereas the difficulty parameter determines the location of the midpoint.

### Interpretation

Increasing the difficulty parameter $b_i$ shifts the logistic curve horizontally.

- A larger $b_i$ shifts the curve to the **right**, meaning a higher ability is required to achieve the same probability of answering correctly.
- A smaller $b_i$ shifts the curve to the **left**, making the item easier.

Changing the discrimination parameter $a_i$ changes the steepness of the curve.

- Larger $a_i$ produces a steeper curve, making the item more sensitive to differences in ability.
- Smaller $a_i$ produces a flatter curve, making the item less discriminative.

In [3]:
import numpy as np
import plotly.graph_objects as go

theta = np.linspace(-4,4,400)

def logistic(theta,a,b):
    return 1/(1+np.exp(-a*(theta-b)))

fig = go.Figure()

# Same discrimination, different difficulty
for b in [-1,0,1]:
    fig.add_trace(
        go.Scatter(
            x=theta,
            y=logistic(theta,1,b),
            mode='lines',
            name=f'a=1, b={b}'
        )
    )

# Different discrimination
fig.add_trace(
    go.Scatter(
        x=theta,
        y=logistic(theta,2,0),
        mode='lines',
        name='a=2, b=0'
    )
)

fig.update_layout(
    title='2PL Item Characteristic Curves',
    xaxis_title='Ability (θ)',
    yaxis_title='P(Y=1|θ)',
    template='plotly_white'
)

fig.show()

###2 - Sequential Likelihood Contribution

For a single observed response

$$
y_k\in\{0,1\},
$$

the likelihood contribution is

$$
L(y_k|\theta)
=
p_k(\theta)^{y_k}
\left(1-p_k(\theta)\right)^{1-y_k},
$$

where

$$
p_k(\theta)
=
\frac{1}{1+e^{-a_k(\theta-b_k)}}.
$$

Assuming that all responses are conditionally independent given the latent ability $\theta$, the joint likelihood for the running response history

$$
y^{(k)}
=
(y_1,y_2,\ldots,y_k)
$$

is

$$
L(y^{(k)}|\theta)
=
\prod_{i=1}^{k}
p_i(\theta)^{y_i}
\left(1-p_i(\theta)\right)^{1-y_i}.
$$

###3 - Mathematical Formulation of the Running Update

Bayes' theorem updates the posterior distribution recursively.

The posterior distribution after observing the $k$-th response is

$$
f_{\Theta|Y^{(k)}}(\theta|y^{(k)})
\propto
L(y_k|\theta)
f_{\Theta|Y^{(k-1)}}(\theta|y^{(k-1)}).
$$

Substituting the likelihood function gives

$$
f_{\Theta|Y^{(k)}}(\theta|y^{(k)})
\propto
p_k(\theta)^{y_k}
\left(1-p_k(\theta)\right)^{1-y_k}
f_{\Theta|Y^{(k-1)}}(\theta|y^{(k-1)}).
$$

Thus, the posterior from the previous step becomes the prior for the next step. Each newly observed response updates the user's estimated ability by multiplying the previous posterior by the likelihood of the new observation before normalization.

###4 - Dynamic Shifting

Suppose the user answers a highly difficult item correctly,

$$
y_k=1,
$$

where the item has a large difficulty parameter $b_k$.

The posterior update becomes

$$
f_{\Theta|Y^{(k)}}(\theta)
\propto
p_k(\theta)
f_{\Theta|Y^{(k-1)}}(\theta).
$$

Since a difficult item has a high value of $b_k$, the probability

$$
p_k(\theta)
=
\frac{1}{1+e^{-a_k(\theta-b_k)}}
$$

becomes large only when $\theta$ is relatively large.

Therefore, multiplying the previous posterior by this likelihood assigns greater probability mass to larger values of $\theta$.

As a result,

- the posterior peak shifts toward higher ability values,
- the posterior mean increases,
- the MAP estimate also increases.

A correct response to a difficult item therefore provides strong evidence that the user possesses a higher latent ability than previously believed.

###5 - Tracking Certainty and Sharpness

The discrimination parameter $a_k$ determines how informative the current item is.

When $a_k$ is large, the Item Characteristic Curve becomes very steep.

A small change in ability then causes a large change in the probability of answering correctly. Consequently, the likelihood function is highly concentrated, resulting in

- a narrower posterior distribution,
- smaller posterior variance,
- greater confidence in the estimated ability.

When $a_k$ is small, the Item Characteristic Curve is flatter.

In this case, responses provide less information about the user's ability. The posterior distribution remains wider, uncertainty decreases more slowly, and the platform gains confidence at a slower rate.

###6 - Numerical Implementation of a Running Grid

A numerical approximation of the posterior distribution can be maintained using a fixed grid of ability values.

The algorithm is as follows:

1. Create a grid of equally spaced $\theta$ values.

2. Initialize the prior distribution using

$$
\Theta\sim N(0,1).
$$

3. For each observed response, evaluate the likelihood

$$
L(y_k|\theta)
$$

at every grid point.

4. Update the posterior by multiplying the previous posterior by the likelihood:

$$
\text{Posterior}
=
\text{Prior}
\times
\text{Likelihood}.
$$

5. Normalize the posterior so that it integrates to one:

$$
f(\theta)
=
\frac{f(\theta)}
{\sum f(\theta)\Delta\theta}.
$$

6. Compute the Posterior Mean

$$
E[\theta]
=
\sum \theta f(\theta)\Delta\theta,
$$

and determine the MAP estimate by locating the maximum value of the posterior density.

7. Repeat the update process after every newly observed response.

###7 - Analysis

In [4]:
import numpy as np
import plotly.graph_objects as go

np.random.seed(42)

# --------------------------
# Parameters
# --------------------------

theta_true = 0.75
n = 20

theta = np.linspace(-4,4,800)
dtheta = theta[1]-theta[0]

prior = np.exp(-0.5*theta**2)
prior /= np.sum(prior*dtheta)

posterior = prior.copy()

posterior_mean = [np.sum(theta*posterior*dtheta)]
MAP = [theta[np.argmax(posterior)]]

# --------------------------
# Sequential Simulation
# --------------------------

for k in range(n):

    a = np.random.uniform(0.5,2.0)
    b = np.random.normal(0,1)

    p_true = 1/(1+np.exp(-a*(theta_true-b)))

    y = np.random.rand() < p_true

    p_theta = 1/(1+np.exp(-a*(theta-b)))

    likelihood = (p_theta**y)*((1-p_theta)**(1-y))

    posterior *= likelihood

    posterior /= np.sum(posterior*dtheta)

    posterior_mean.append(np.sum(theta*posterior*dtheta))

    MAP.append(theta[np.argmax(posterior)])

# --------------------------
# Plot
# --------------------------

steps = np.arange(n+1)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=steps,
        y=posterior_mean,
        mode='lines+markers',
        name='Posterior Mean'
    )
)

fig.add_trace(
    go.Scatter(
        x=steps,
        y=MAP,
        mode='lines+markers',
        name='MAP'
    )
)

fig.add_hline(
    y=theta_true,
    line_dash='dash',
    annotation_text='True Ability'
)

fig.update_layout(
    title='Running Bayesian Ability Estimation',
    xaxis_title='Item Number',
    yaxis_title='Ability Estimate',
    template='plotly_white'
)

fig.show()

/tmp/ipykernel_3513/4151262291.py:39: DeprecationWarning:

In future, it will be an error for 'np.bool' scalars to be interpreted as an index





The figure shows the progression of both the Posterior Mean and the Maximum A Posteriori (MAP) estimate as more item responses are observed.

Initially, both estimators are heavily influenced by the prior distribution because only a few responses are available. Consequently, the estimates may fluctuate considerably and differ from the true ability value,

$$
\theta_{\text{true}}=0.75.
$$

As additional responses are incorporated through Bayesian updating, the likelihood accumulates more evidence regarding the user's ability. Both estimators gradually converge toward the true latent ability.

At the same time, the posterior distribution becomes increasingly concentrated around the estimated ability, indicating a reduction in posterior variance. This reduction in variance reflects increasing confidence in the user's estimated ability.

Overall, the decreasing distance between the Posterior Mean, the MAP estimate, and the true ability demonstrates that sequential Bayesian updating enables increasingly accurate and reliable estimation as more observations become available.

# Question 2
###Structural Probability and Properties

The Beta distribution is commonly used as a prior distribution for an unknown probability parameter because its support is restricted to the interval $[0,1]$.

The probability density function (PDF) of a Beta distribution is

$$
f(\theta)=\frac{1}{B(\alpha,\beta)}
\theta^{\alpha-1}(1-\theta)^{\beta-1},
\qquad 0\le\theta\le1,
$$

where

- $\alpha>0$ and $\beta>0$ are the shape parameters,
- $B(\alpha,\beta)$ is the Beta function acting as the normalization constant.

The following parameter sets illustrate different prior beliefs:

- **Beta(1,1):** Uniform (uninformative prior)
- **Beta(2,8):** Right-skewed distribution
- **Beta(8,2):** Left-skewed distribution

### Interpretation

The parameters $\alpha$ and $\beta$ determine where the probability mass is concentrated.

- When $\alpha=\beta=1$, the density is uniform over $[0,1]$, indicating no prior preference for any click-through rate.
- When $\alpha<\beta$, the density is concentrated near 0, reflecting the belief that the click probability is likely to be small.
- When $\alpha>\beta$, the density is concentrated near 1, indicating a belief that the click probability is relatively high.

Thus, increasing $\alpha$ shifts the center of mass toward larger values of $\theta$, while increasing $\beta$ shifts it toward smaller values.

In [6]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

theta = np.linspace(0,1,500)

fig = go.Figure()

params=[(1,1),(2,8),(8,2)]

for a,b in params:
    fig.add_trace(
        go.Scatter(
            x=theta,
            y=beta.pdf(theta,a,b),
            mode='lines',
            name=f'Beta({a},{b})'
        )
    )

fig.update_layout(
    title='Beta Distribution PDFs',
    xaxis_title='θ',
    yaxis_title='Density',
    template='plotly_white'
)

fig.show()

### 2. Sequential Likelihood and Joint History

Each user interaction is modeled as an independent Bernoulli trial.

For a single observation

$$
y_k\in\{0,1\},
$$

the likelihood contribution is

$$
L(y_k|\theta)
=
\theta^{y_k}(1-\theta)^{1-y_k}.
$$

Assuming conditional independence of user interactions, the joint likelihood for the response history

$$
y^{(k)}
=
(y_1,y_2,\ldots,y_k)
$$

is

$$
L(y^{(k)}|\theta)
=
\prod_{i=1}^{k}
\theta^{y_i}(1-\theta)^{1-y_i}.
$$

Let

$$
S_k=\sum_{i=1}^{k}y_i
$$

denote the total number of clicks observed so far.

The joint likelihood can therefore be simplified as

$$
L(y^{(k)}|\theta)
=
\theta^{S_k}
(1-\theta)^{k-S_k}.
$$

###3. Closed-Form Analytical Updates (Conjugacy)

Assume the prior distribution is

$$
\Theta\sim\text{Beta}(\alpha_{k-1},\beta_{k-1}),
$$

with density

$$
f(\theta)
=
\frac{1}{B(\alpha_{k-1},\beta_{k-1})}
\theta^{\alpha_{k-1}-1}
(1-\theta)^{\beta_{k-1}-1}.
$$

Using Bayes' theorem,

$$
f(\theta|y^{(k)})
\propto
L(y_k|\theta)
f(\theta|y^{(k-1)}).
$$

Substituting the Bernoulli likelihood,

$$
f(\theta|y^{(k)})
\propto
\theta^{y_k}
(1-\theta)^{1-y_k}
\theta^{\alpha_{k-1}-1}
(1-\theta)^{\beta_{k-1}-1}.
$$

Combining the exponents,

$$
f(\theta|y^{(k)})
\propto
\theta^{\alpha_{k-1}+y_k-1}
(1-\theta)^{\beta_{k-1}+1-y_k-1}.
$$

Hence,

$$
\boxed{
\Theta|Y^{(k)}
\sim
\text{Beta}(\alpha_k,\beta_k)
}
$$

where

$$
\boxed{
\alpha_k=\alpha_{k-1}+y_k
}
$$

and

$$
\boxed{
\beta_k=\beta_{k-1}+1-y_k.
}
$$

Therefore, the Beta family is conjugate to the Bernoulli likelihood.

The posterior mean is

$$
\boxed{
E[\Theta|Y^{(k)}]
=
\frac{\alpha_k}{\alpha_k+\beta_k}.
}
$$

###4. Dynamic Shifting Mechanics

The posterior parameters are updated analytically after each observation.

If a click is observed,

$$
y_k=1,
$$

then

$$
\alpha_k=\alpha_{k-1}+1,
$$

while

$$
\beta_k=\beta_{k-1}.
$$

Consequently, the posterior density shifts toward larger values of $\theta$, increasing both the posterior mean and the posterior mode.

If a non-click is observed,

$$
y_k=0,
$$

then

$$
\beta_k=\beta_{k-1}+1,
$$

while

$$
\alpha_k=\alpha_{k-1}.
$$

The posterior density therefore shifts toward smaller values of $\theta$.

Unlike the Beta-Binomial model, which admits closed-form posterior updates, non-conjugate Bayesian models such as the Two-Parameter Logistic (2PL) Item Response Theory model do not produce a posterior distribution of the same functional form. Consequently, analytical updates are generally impossible, and numerical techniques such as grid approximation, numerical integration, Markov Chain Monte Carlo (MCMC), or variational inference must be employed to approximate the posterior distribution.

###5. Running Point Estimators

After updating the shape parameters, the running Bayesian estimators can be computed directly.

### Posterior Mean

The posterior mean is

$$
\boxed{
\hat{\theta}_{Bayes}^{(k)}
=
\frac{\alpha_k}
{\alpha_k+\beta_k}.
}
$$

### Maximum A Posteriori (MAP)

For

$$
\alpha_k>1,\qquad
\beta_k>1,
$$

the posterior mode is

$$
\boxed{
\hat{\theta}_{MAP}^{(k)}
=
\frac{\alpha_k-1}
{\alpha_k+\beta_k-2}.
}
$$

If either parameter is less than or equal to one, the Beta distribution has its maximum at the boundary of the interval $[0,1]$, and the MAP estimate is obtained accordingly.

In [7]:
import numpy as np
import plotly.graph_objects as go

np.random.seed(42)

theta_true=0.35
n=100

alpha=1
beta=1

posterior_mean=[]
MAP=[]

posterior_mean.append(alpha/(alpha+beta))
MAP.append(np.nan)

for k in range(n):

    y=np.random.rand()<theta_true

    alpha+=y
    beta+=1-y

    posterior_mean.append(alpha/(alpha+beta))

    if alpha>1 and beta>1:
        MAP.append((alpha-1)/(alpha+beta-2))
    else:
        MAP.append(np.nan)

steps=np.arange(n+1)

fig=go.Figure()

fig.add_trace(
    go.Scatter(
        x=steps,
        y=posterior_mean,
        mode='lines',
        name='Posterior Mean'
    )
)

fig.add_trace(
    go.Scatter(
        x=steps,
        y=MAP,
        mode='lines',
        name='MAP'
    )
)

fig.add_hline(
    y=theta_true,
    line_dash='dash',
    annotation_text='True CTR'
)

fig.update_layout(
    title='Sequential Beta-Binomial Estimation',
    xaxis_title='Impression Number',
    yaxis_title='Estimated CTR',
    template='plotly_white'
)

fig.show()

###6. Analysis

The figure illustrates how both the Posterior Mean and the Maximum A Posteriori (MAP) estimate evolve as additional user interactions are observed.

Initially, the estimators are strongly influenced by the prior distribution because only a limited number of observations are available. Consequently, both estimates may differ noticeably from the true click-through rate,

$$
\theta_{\mathrm{true}}=0.35.
$$

As the number of impressions increases, the influence of the observed data accumulates through repeated Bayesian updates. Both the Posterior Mean and the MAP estimate gradually converge toward the true click-through rate.

By the time the number of impressions approaches 100, the distance between the estimators and the true parameter becomes much smaller. This indicates that the posterior distribution is becoming increasingly concentrated, reflecting greater certainty about the advertisement's true click probability.

The convergence also demonstrates an important property of Bayesian inference: although the initial prior influences the estimates during the early stages, its effect diminishes as more evidence is collected. Eventually, the observed data dominate the posterior distribution, resulting in stable and accurate estimates of the click-through rate.

# Question 3 - Bayesian Estimations for Structural Health Monitoring via Bounded Grid Updates
###1. Prior Belief Boundaries

Before any sensor measurements are collected, engineers assume that the structural component is most likely in a healthy condition. This prior belief is modeled using the Beta distribution

$$
\Theta \sim \text{Beta}(8,1.5),
$$

whose probability density function is

$$
f_{\Theta}(\theta)
=
\frac{1}{B(8,1.5)}
\theta^{8-1}
(1-\theta)^{1.5-1},
\qquad
0<\theta\le1,
$$

where

- $\theta$ represents the remaining structural stiffness efficiency,
- $B(\cdot,\cdot)$ is the Beta function.

The expected value of a Beta distribution is

$$
E[\Theta]
=
\frac{\alpha}{\alpha+\beta}.
$$

Therefore,

$$
E[\Theta^{(0)}]
=
\frac{8}{8+1.5}
=
\frac{8}{9.5}
=
0.8421.
$$

Thus, before observing any sensor measurements, engineers expect the structure to retain approximately **84.2%** of its nominal stiffness.

### Interpretation

The Beta(8,1.5) distribution places most of its probability mass close to $\theta=1$, indicating a strong prior belief that the component is healthy.

This choice is appropriate because newly manufactured or recently inspected engineering structures are generally expected to operate close to their nominal stiffness. At the same time, the distribution still allows lower stiffness values, enabling the Bayesian updating process to detect unexpected structural degradation when new sensor measurements become available.

In [8]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

theta = np.linspace(0.01,1.0,500)

pdf = beta.pdf(theta,8,1.5)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=theta,
        y=pdf,
        mode='lines',
        name='Beta(8,1.5)'
    )
)

fig.update_layout(
    title='Initial Prior Distribution',
    xaxis_title='Remaining Stiffness Efficiency (θ)',
    yaxis_title='Probability Density',
    template='plotly_white'
)

fig.show()

###2. Structural Likelihood Formulation

The structural sensor follows the measurement model

$$
y_k
=
\theta K_{\mathrm{nominal}}
e^{\varepsilon_k},
$$

where

$$
\varepsilon_k
\sim
N(0,\sigma^2).
$$

Taking the natural logarithm,

$$
\ln(y_k)
=
\ln(\theta K_{\mathrm{nominal}})
+
\varepsilon_k.
$$

Therefore,

$$
\ln(y_k)
\sim
N
\left(
\ln(\theta K_{\mathrm{nominal}}),
\sigma^2
\right),
$$

which implies that the measurement follows a Log-Normal distribution.

The likelihood contribution of one sensor measurement is therefore

$$
L(y_k|\theta)
=
\frac{1}
{y_k\sigma\sqrt{2\pi}}
\exp
\left[
-\frac{
\left(
\ln y_k-
\ln(\theta K_{\mathrm{nominal}})
\right)^2
}
{2\sigma^2}
\right].
$$

Assuming conditional independence of sensor measurements, the joint likelihood for the measurement history

$$
y^{(k)}
=
(y_1,y_2,\ldots,y_k)
$$

is

$$
L(y^{(k)}|\theta)
=
\prod_{i=1}^{k}
L(y_i|\theta).
$$

####3. Mathematical Formulation of the Non-Conjugate Grid Update

The prior distribution is Beta distributed,

$$
\Theta
\sim
\text{Beta}(8,1.5),
$$

whereas the likelihood is Log-Normally distributed.

Unlike the Beta-Bernoulli or Beta-Binomial model, these two distributions are **not conjugate**.

Consequently, multiplying the Beta prior by the Log-Normal likelihood does not produce another distribution belonging to any known probability family. Therefore, an exact closed-form analytical posterior does not exist.

Instead, Bayes' theorem is evaluated numerically.

The recursive Bayesian update is

$$
f_{\Theta|Y^{(k)}}(\theta|y^{(k)})
\propto
L(y_k|\theta)
f_{\Theta|Y^{(k-1)}}(\theta|y^{(k-1)}).
$$

where

$$
L(y_k|\theta)
=
\frac{1}
{y_k\sigma\sqrt{2\pi}}
\exp
\left[
-\frac{
\left(
\ln y_k-
\ln(\theta K_{\mathrm{nominal}})
\right)^2
}
{2\sigma^2}
\right].
$$

After each observation, the resulting posterior density must be normalized numerically before being used as the prior for the next inspection step.

###4. Running Point Estimates

Since the posterior distribution has no closed-form expression, numerical integration is required to compute the Bayesian estimators.

### Running Posterior Mean

The posterior mean is

$$
\boxed{
\hat{\theta}^{(k)}_{\mathrm{Bayes}}
=
\int_0^1
\theta
f_{\Theta|Y^{(k)}}(\theta)
\,d\theta.
}
$$

The integral is evaluated numerically using the posterior density defined on a discrete grid.

### Running Maximum A Posteriori (MAP)

The MAP estimate is defined as

$$
\boxed{
\hat{\theta}^{(k)}_{\mathrm{MAP}}
=
\underset{0<\theta\le1}{\operatorname{argmax}}
\;
f_{\Theta|Y^{(k)}}(\theta).
}
$$

Numerically, this corresponds to locating the grid point having the largest posterior probability density.

###5. Algorithmic Grid Approximation and Normalization

Since no analytical posterior exists, the posterior distribution is approximated on a fixed grid of stiffness values.

The numerical procedure is as follows.

1. Construct a grid of equally spaced values

$$
\theta
\in
[0.01,1.00].
$$

The lower limit is chosen slightly above zero to avoid numerical issues associated with taking logarithms and dividing by zero.

2. Evaluate the initial Beta prior density at every grid point.

3. After receiving a new sensor measurement $y_k$, evaluate the Log-Normal likelihood at every grid point.

4. Compute the unnormalized posterior

$$
f_{\mathrm{new}}(\theta)
=
f_{\mathrm{old}}(\theta)
\times
L(y_k|\theta).
$$

5. Normalize the posterior using the trapezoidal rule

$$
f(\theta)
=
\frac{
f(\theta)
}
{
\displaystyle
\int_{0.01}^{1}
f(\theta)
\,d\theta
}.
$$

Computationally,

```python
posterior /= np.trapezoid(posterior, theta_grid)
```

ensures that the posterior integrates to one.

6. Compute the Posterior Mean using numerical integration,

$$
\hat{\theta}_{Bayes}
=
\int_0^1
\theta
f(\theta)
\,d\theta,
$$

and compute the MAP estimate by selecting the grid point corresponding to the maximum posterior density.

7. Repeat the Bayesian update after every new structural sensor measurement.

###6. Analysis

In [10]:
# Approximate convergence point (within ±0.02 of the true value)
for i, est in enumerate(bayes_est):
    if abs(est - theta_true) < 0.02:
        print(f"Bayes estimate first comes within ±0.02 of the true value at step {i}.")
        break

Bayes estimate first comes within ±0.02 of the true value at step 15.


In [9]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

# -----------------------------
# Parameters
# -----------------------------

np.random.seed(42)

theta_true = 0.68
K_nominal = 50.0          # kN/mm
sigma = 0.15
n = 15

# -----------------------------
# Grid
# -----------------------------

theta_grid = np.linspace(0.01,1.0,800)

# Initial Prior: Beta(8,1.5)

posterior = beta.pdf(theta_grid,8,1.5)
posterior /= np.trapezoid(posterior,theta_grid)

# -----------------------------
# Store Results
# -----------------------------

posterior_history = {0:posterior.copy()}

bayes_est = [
    np.trapezoid(theta_grid*posterior,theta_grid)
]

map_est = [
    theta_grid[np.argmax(posterior)]
]

# -----------------------------
# Sequential Measurements
# -----------------------------

for k in range(1,n+1):

    # Simulated sensor measurement
    noise = np.random.normal(0,sigma)

    y = theta_true*K_nominal*np.exp(noise)

    # Log-normal likelihood
    likelihood = (
        1/(y*sigma*np.sqrt(2*np.pi))
    )*np.exp(
        -(
            np.log(y)-np.log(theta_grid*K_nominal)
        )**2/(2*sigma**2)
    )

    # Bayesian update

    posterior *= likelihood

    posterior /= np.trapezoid(posterior,theta_grid)

    # Save selected posterior curves

    if k in [1,2,5,10,15]:
        posterior_history[k]=posterior.copy()

    # Posterior Mean

    bayes = np.trapezoid(
        theta_grid*posterior,
        theta_grid
    )

    bayes_est.append(bayes)

    # MAP

    map_est.append(
        theta_grid[np.argmax(posterior)]
    )

# -----------------------------
# Plot 1
# Posterior Densities
# -----------------------------

fig = go.Figure()

for k in [0,1,2,5,10,15]:

    fig.add_trace(
        go.Scatter(
            x=theta_grid,
            y=posterior_history[k],
            mode="lines",
            name=f"k={k}"
        )
    )

fig.update_layout(
    title="Evolution of Posterior Density",
    xaxis_title="Remaining Stiffness θ",
    yaxis_title="Posterior Density",
    template="plotly_white"
)

fig.show()

# -----------------------------
# Plot 2
# Running Estimates
# -----------------------------

steps=np.arange(n+1)

fig=go.Figure()

fig.add_trace(
    go.Scatter(
        x=steps,
        y=bayes_est,
        mode="lines+markers",
        name="Posterior Mean"
    )
)

fig.add_trace(
    go.Scatter(
        x=steps,
        y=map_est,
        mode="lines+markers",
        name="MAP"
    )
)

fig.add_hline(
    y=theta_true,
    line_dash="dash",
    annotation_text="True Stiffness"
)

fig.update_layout(
    title="Sequential Bayesian Structural Health Monitoring",
    xaxis_title="Inspection Step",
    yaxis_title="Estimated Remaining Stiffness",
    template="plotly_white"
)

fig.show()

The posterior density evolves significantly as additional structural sensor measurements are incorporated.

Initially, the posterior distribution is concentrated near high stiffness values because the prior distribution Beta(8,1.5) strongly reflects the engineering belief that newly deployed structures are likely to be healthy. Consequently, the initial Posterior Mean and MAP estimates are both substantially higher than the true remaining stiffness,

$$
\theta_{\mathrm{true}}=0.68.
$$

As more vibration measurements are collected, the likelihood repeatedly shifts the posterior distribution toward lower stiffness values that better explain the observed data. After only a few measurements, the influence of the optimistic prior begins to diminish, and both the Posterior Mean and MAP estimate move steadily toward the true stiffness factor.

By approximately **5 to 8 sensor measurements**, the posterior has largely overcome the initial healthy prior and becomes centered close to the true value of 0.68. As additional measurements are incorporated, both estimators stabilize and fluctuate only slightly around the true stiffness.

The posterior density also becomes progressively narrower throughout the monitoring process. This reduction in spread indicates decreasing uncertainty regarding the structural condition and increasing confidence in the estimated stiffness. From an engineering perspective, a narrower posterior distribution allows more reliable assessment of structural safety margins and enables maintenance decisions to be made with greater confidence.

Overall, the simulation demonstrates how Bayesian sequential updating combines prior engineering knowledge with continuously arriving sensor data to provide an increasingly accurate estimate of structural health while naturally quantifying uncertainty throughout the monitoring process.

#4. Gaussian Mixture Clustering as Conditional Updating

###1. Deriving the Marginal Density

Let the latent cluster variable be

$$
C_i \in \{1,2,\ldots,K\},
$$

where

$$
P(C_i=k)=\phi_k,
$$

with

$$
\phi_k \ge 0,
\qquad
\sum_{k=1}^{K}\phi_k=1.
$$

Conditional on the cluster assignment,

$$
X_i|C_i=k
\sim
\mathcal{N}(\mu_k,\Sigma_k).
$$

Using the Law of Total Probability,

$$
p(x_i)
=
\sum_{k=1}^{K}
P(C_i=k)
\,p(x_i|C_i=k).
$$

Substituting the Gaussian model,

$$
p(x_i)
=
\sum_{k=1}^{K}
\phi_k
\,
\mathcal{N}
(x_i|\mu_k,\Sigma_k).
$$

Therefore,

$$
\boxed{
p(x_i)
=
\sum_{k=1}^{K}
\phi_k
\mathcal{N}(x_i|\mu_k,\Sigma_k).
}
$$

### Interpretation

The marginal density is called a **Gaussian Mixture Density** because it is formed by combining multiple Gaussian distributions, each weighted by its corresponding prior probability.

Each Gaussian component represents one cluster, while the mixture weights determine how much each cluster contributes to the overall probability density.

Consequently, the overall distribution can represent complex, multi-modal datasets that cannot be modeled adequately using a single Gaussian distribution.

###2. Deriving the Posterior Cluster Probability

For an observed data point $x_i$, Bayes' theorem gives

$$
P(C_i=k|X_i=x_i)
=
\frac{
P(X_i=x_i|C_i=k)
P(C_i=k)
}
{
\sum_{j=1}^{K}
P(X_i=x_i|C_i=j)
P(C_i=j)
}.
$$

Substituting the Gaussian likelihood and the prior cluster probabilities,

$$
P(C_i=k|X_i=x_i)
=
\frac{
\phi_k
\mathcal{N}(x_i|\mu_k,\Sigma_k)
}
{
\sum_{j=1}^{K}
\phi_j
\mathcal{N}(x_i|\mu_j,\Sigma_j)
}.
$$

This posterior probability is called the **responsibility** of cluster $k$ for observation $x_i$ and is denoted by

$$
\boxed{
\gamma_{ik}
=
P(C_i=k|X_i=x_i).
}
$$

### Interpretation

The responsibility represents the probability that observation $x_i$ belongs to cluster $k$ after observing the data.

Before observing $x_i$, only the prior probabilities

$$
\phi_k
$$

are known.

After incorporating the observed data through Bayes' theorem, these prior beliefs are updated into posterior probabilities,

$$
\gamma_{ik},
$$

which quantify the likelihood that each cluster generated the observation.

Since

$$
0\le\gamma_{ik}\le1,
\qquad
\sum_{k=1}^{K}\gamma_{ik}=1,
$$

the responsibilities form a valid probability distribution over all clusters.

##3. One-Hot Encoding of the Latent Cluster Variable

Define the one-hot encoded latent vector

$$
Z_i=
\begin{bmatrix}
Z_{i1}\\
Z_{i2}\\
\vdots\\
Z_{iK}
\end{bmatrix},
$$

where

$$
Z_{ik}
=
\begin{cases}
1,&\text{if }C_i=k,\\
0,&\text{otherwise}.
\end{cases}
$$

Since $Z_{ik}$ is a Bernoulli random variable,

$$
E[Z_{ik}|X_i=x_i]
=
1\cdot P(C_i=k|X_i=x_i)
+
0\cdot P(C_i\neq k|X_i=x_i).
$$

Therefore,

$$
\boxed{
E[Z_{ik}|X_i=x_i]
=
P(C_i=k|X_i=x_i)
=
\gamma_{ik}.
}
$$

Hence,

$$
E[Z_i|X_i=x_i]
=
\begin{bmatrix}
\gamma_{i1}\\
\gamma_{i2}\\
\vdots\\
\gamma_{iK}
\end{bmatrix}.
$$

Therefore,

$$
\boxed{
E[Z_i|X_i=x_i]
=
\begin{bmatrix}
\gamma_{i1}\\
\gamma_{i2}\\
\vdots\\
\gamma_{iK}
\end{bmatrix}.
}
$$

### Interpretation

The conditional expectation of the latent indicator vector is exactly the vector of posterior cluster probabilities.

Rather than assigning the observation to a single cluster, the model distributes its membership probabilistically across all clusters.

This vector therefore represents the **soft cluster assignment** of the observation.

###4. From Soft Assignment to Hard Clustering

The soft assignment of observation $x_i$ is given by

$$
E[Z_i|X_i=x_i]
=
\begin{bmatrix}
\gamma_{i1}\\
\gamma_{i2}\\
\vdots\\
\gamma_{iK}
\end{bmatrix},
$$

where each responsibility represents the posterior probability that the observation belongs to a particular cluster.

A hard cluster assignment is obtained by selecting the cluster with the largest posterior probability,

$$
\boxed{
\hat{C}_i
=
\operatorname*{arg\,max}_{1\le k\le K}
\gamma_{ik}.
}
$$

### Soft Clustering

In soft clustering, every observation belongs to every cluster with different probabilities.

For example,

$$
E[Z_i|X_i=x_i]
=
\begin{bmatrix}
0.70\\
0.20\\
0.10
\end{bmatrix}
$$

indicates that the observation has

- 70% probability of belonging to Cluster 1,
- 20% probability of belonging to Cluster 2,
- 10% probability of belonging to Cluster 3.

This approach preserves uncertainty and is particularly useful when observations lie near cluster boundaries.

### Hard Clustering

In hard clustering, the observation is assigned only to the cluster having the largest posterior probability.

For the above example,

$$
\hat{C}_i=1.
$$

Thus, hard clustering converts the probabilistic soft assignment into a single deterministic cluster label.

Compared with hard clustering, soft clustering provides richer information by quantifying the uncertainty associated with cluster membership.

###5. Conditional Expectation of the Observation Given the Cluster

Recall that, conditional on belonging to cluster $k$, the observation follows a multivariate Gaussian distribution,

$$
X_i \mid C_i=k
\sim
\mathcal{N}(\mu_k,\Sigma_k).
$$

The expectation of a multivariate Gaussian random variable is simply its mean vector. Therefore,

$$
\boxed{
E[X_i \mid C_i=k]
=
\mu_k.
}
$$

Hence, the vector $\mu_k$ represents the **center** of cluster $k$ because it is the average location of all observations generated by that cluster.

The two conditional expectations in a Gaussian Mixture Model have different meanings.

The first is

$$
E[Z_i \mid X_i=x_i]
=
\begin{bmatrix}
\gamma_{i1}\\
\gamma_{i2}\\
\vdots\\
\gamma_{iK}
\end{bmatrix},
$$

which gives the posterior probabilities that an observed point belongs to each cluster. It therefore describes the **soft cluster membership** of an individual observation.

The second is

$$
E[X_i \mid C_i=k]
=
\mu_k,
$$

which gives the expected location of observations belonging to cluster $k$. It therefore describes the geometric center of the cluster.

Thus,

- $E[Z_i|X_i=x_i]$ provides probabilistic cluster membership for an observation.
- $E[X_i|C_i=k]$ provides the mean location of an entire cluster.

###6. The Complete-Data Likelihood

Suppose the latent cluster indicators

$$
z_{ik}
$$

are known.

The complete-data likelihood is

$$
p(x_1,\ldots,x_n,z_1,\ldots,z_n)
=
\prod_{i=1}^{n}
\prod_{k=1}^{K}
\left[
\phi_k
\mathcal{N}(x_i|\mu_k,\Sigma_k)
\right]^{z_{ik}}.
$$

Taking the natural logarithm,

$$
\ell_c
=
\log p(x_1,\ldots,x_n,z_1,\ldots,z_n).
$$

Using the logarithm of a product,

$$
\log(ab)=\log a+\log b,
$$

gives

$$
\boxed{
\ell_c
=
\sum_{i=1}^{n}
\sum_{k=1}^{K}
z_{ik}
\left[
\log\phi_k
+
\log
\mathcal{N}(x_i|\mu_k,\Sigma_k)
\right].
}
$$

### Interpretation

If the values of $z_{ik}$ were known, every observation would already have a known cluster assignment.

The optimization problem would therefore separate naturally into independent calculations for each cluster.

Consequently, estimating the mixture weights, cluster means, and covariance matrices becomes straightforward because each observation contributes only to its assigned cluster.

###7. The EM Interpretation

In practice, the latent variables

$$
z_{ik}
$$

are not observed.

Instead, the Expectation-Maximization (EM) algorithm replaces each unknown indicator by its conditional expectation,

$$
z_{ik}
\longrightarrow
E[Z_{ik}|X_i=x_i].
$$

Since

$$
E[Z_{ik}|X_i=x_i]
=
\gamma_{ik},
$$

the expected complete-data log-likelihood becomes

$$
\boxed{
Q
=
\sum_{i=1}^{n}
\sum_{k=1}^{K}
\gamma_{ik}
\left[
\log\phi_k
+
\log
\mathcal{N}(x_i|\mu_k,\Sigma_k)
\right].
}
$$

### Interpretation

The E-step computes the posterior probabilities

$$
\gamma_{ik},
$$

which quantify the probability that each observation belongs to every cluster.

Rather than making a hard assignment, each observation contributes fractionally to all clusters according to these probabilities.

Therefore, the E-step can be interpreted as a **conditional Bayesian update** of cluster membership probabilities after observing the data.

###8. Parameter Updates

After computing the responsibilities, the M-step updates the model parameters by maximizing the expected complete-data log-likelihood.

First, compute the effective number of observations assigned to cluster $k$,

$$
\boxed{
N_k
=
\sum_{i=1}^{n}
\gamma_{ik}.
}
$$

The updated mixture weight is

$$
\boxed{
\phi_k^{\text{new}}
=
\frac{N_k}{n}.
}
$$

The updated cluster mean is

$$
\boxed{
\mu_k^{\text{new}}
=
\frac{1}{N_k}
\sum_{i=1}^{n}
\gamma_{ik}x_i.
}
$$

The updated covariance matrix is

$$
\boxed{
\Sigma_k^{\text{new}}
=
\frac{1}{N_k}
\sum_{i=1}^{n}
\gamma_{ik}
(x_i-\mu_k^{\text{new}})
(x_i-\mu_k^{\text{new}})^T.
}
$$

### Interpretation

The responsibility

$$
\gamma_{ik}
$$

acts as a **fractional membership weight**.

Instead of contributing completely to a single cluster, each observation contributes partially to every cluster according to its posterior probability.

Observations with large responsibilities have greater influence on the parameter estimates, while observations with small responsibilities contribute only slightly.

This weighted averaging allows the Gaussian Mixture Model to represent overlapping clusters naturally.

###9. Interpretation

Gaussian Mixture Model clustering can be viewed as a repeated process of conditional Bayesian updating.

Initially, each cluster has a prior probability given by the mixture weight

$$
\phi_k,
$$

which represents the probability that a randomly selected observation belongs to cluster $k$ before any data are observed.

For an observed point $x_i$, the Gaussian density

$$
\mathcal{N}(x_i|\mu_k,\Sigma_k)
$$

measures how compatible the observation is with cluster $k$. Combining this likelihood with the prior through Bayes' theorem produces the posterior probability

$$
\gamma_{ik}
=
P(C_i=k|X_i=x_i),
$$

known as the responsibility.

The vector

$$
E[Z_i|X_i=x_i]
=
\begin{bmatrix}
\gamma_{i1}\\
\gamma_{i2}\\
\vdots\\
\gamma_{iK}
\end{bmatrix}
$$

therefore represents the soft assignment of the observation across all clusters.

During the M-step, these posterior probabilities are used as fractional weights to update the mixture weights, cluster means, and covariance matrices. The updated parameters are then used to compute new posterior probabilities during the next E-step.

Consequently, Gaussian Mixture Model clustering is a probabilistic clustering technique in which latent cluster memberships are repeatedly updated through conditional expectations until the model converges.

In [28]:
# ==========================================================
# Gaussian Mixture Model for Financial Customer Segmentation
# Part 3A
# ==========================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture

import plotly.express as px
import plotly.graph_objects as go

class GMMFinancialSegmenter:

    def __init__(self,
                 csv_path,
                 features=["PURCHASES","CREDIT_LIMIT"],
                 n_components=3,
                 random_state=42):

        self.csv_path = csv_path
        self.features = features
        self.n_components = n_components
        self.random_state = random_state

        self.scaler = StandardScaler()

        self.model = GaussianMixture(
            n_components=n_components,
            covariance_type="full",
            random_state=random_state
        )

    # -------------------------------------------------------
    # Load Dataset
    # -------------------------------------------------------

    def load_data(self):

        self.df = pd.read_csv(self.csv_path)

        self.df = self.df[self.features].dropna()

        print("Dataset Shape :", self.df.shape)

        return self.df

    # -------------------------------------------------------
    # Prepare Data
    # -------------------------------------------------------

    def prepare_data(self):

        X = self.df.values

        X_scaled = self.scaler.fit_transform(X)

        (
            self.X_train,
            self.X_test
        ) = train_test_split(
            X_scaled,
            test_size=0.20,
            random_state=self.random_state
        )

        print("\nTraining Samples :", len(self.X_train))
        print("Testing Samples  :", len(self.X_test))

    # -------------------------------------------------------
    # Fit Gaussian Mixture Model
    # -------------------------------------------------------

    def fit_model(self):

        self.model.fit(self.X_train)

        print("\n========== EM RESULTS ==========")

        print("Converged :",
              self.model.converged_)

        print("Iterations :",
              self.model.n_iter_)

        print("\nMixture Weights")

        print(self.model.weights_)

        print("\nMeans")

        print(self.model.means_)

        print("\nCovariance Matrices")

        for i, cov in enumerate(self.model.covariances_):

            print(f"\nCluster {i+1}")

            print(cov)

    # -------------------------------------------------------
    # Evaluate Model
    # -------------------------------------------------------

    def evaluate_model(self):

        train_score = self.model.score(self.X_train)

        test_score = self.model.score(self.X_test)

        print("\n========== MODEL PERFORMANCE ==========")

        print(f"Average Train Log-Likelihood : {train_score:.4f}")

        print(f"Average Test Log-Likelihood  : {test_score:.4f}")

        self.train_labels = self.model.predict(self.X_train)

        self.test_labels = self.model.predict(self.X_test)

        self.train_resp = self.model.predict_proba(
            self.X_train
        )

        self.test_resp = self.model.predict_proba(
            self.X_test
        )
            # -------------------------------------------------------
    # Plot 1 - Empirical Density Heatmap
    # -------------------------------------------------------

    def plot_density_heatmap(self):

        df_train = pd.DataFrame(
            self.X_train,
            columns=self.features
        )

        fig = px.density_heatmap(
            df_train,
            x=self.features[0],
            y=self.features[1],
            marginal_x="histogram",
            marginal_y="histogram",
            nbinsx=40,
            nbinsy=40,
            title="Training Data Density"
        )

        fig.show()

    # -------------------------------------------------------
    # Plot 2 - Training Assignment
    # -------------------------------------------------------

    def plot_training_assignment(self):

        x_min = self.X_train[:,0].min()-1
        x_max = self.X_train[:,0].max()+1

        y_min = self.X_train[:,1].min()-1
        y_max = self.X_train[:,1].max()+1

        xx, yy = np.meshgrid(
            np.linspace(x_min,x_max,250),
            np.linspace(y_min,y_max,250)
        )

        grid = np.c_[xx.ravel(),yy.ravel()]

        gamma = self.model.predict_proba(grid)

        z = np.max(gamma,axis=1)

        fig = go.Figure()

        fig.add_trace(
            go.Contour(
                x=np.linspace(x_min,x_max,250),
                y=np.linspace(y_min,y_max,250),
                z=z.reshape(xx.shape),
                colorscale="Viridis",
                opacity=0.75,
                showscale=True
            )
        )

        fig.add_trace(
            go.Scatter(
                x=self.X_train[:,0],
                y=self.X_train[:,1],
                mode="markers",
                marker=dict(
                    color=self.train_labels,
                    colorscale="Set1",
                    size=6
                ),
                name="Training Data"
            )
        )

        fig.update_layout(
            title="Training Assignment and Posterior Responsibilities",
            xaxis_title=self.features[0],
            yaxis_title=self.features[1]
        )

        fig.show()

    # -------------------------------------------------------
    # Plot 3 - Test Assignment
    # -------------------------------------------------------

    def plot_test_assignment(self):

        x_min = self.X_train[:,0].min()-1
        x_max = self.X_train[:,0].max()+1

        y_min = self.X_train[:,1].min()-1
        y_max = self.X_train[:,1].max()+1

        xx, yy = np.meshgrid(
            np.linspace(x_min,x_max,250),
            np.linspace(y_min,y_max,250)
        )

        grid = np.c_[xx.ravel(),yy.ravel()]

        gamma = self.model.predict_proba(grid)

        z = np.max(gamma,axis=1)

        fig = go.Figure()

        fig.add_trace(
            go.Contour(
                x=np.linspace(x_min,x_max,250),
                y=np.linspace(y_min,y_max,250),
                z=z.reshape(xx.shape),
                colorscale="Viridis",
                opacity=0.75,
                showscale=True
            )
        )

        fig.add_trace(
            go.Scatter(
                x=self.X_test[:,0],
                y=self.X_test[:,1],
                mode="markers",
                marker=dict(
                    color=self.test_labels,
                    colorscale="Set1",
                    size=6
                ),
                name="Test Data"
            )
        )

        fig.update_layout(
            title="Out-of-Sample Test Assignment",
            xaxis_title=self.features[0],
            yaxis_title=self.features[1]
        )

        fig.show()

    # -------------------------------------------------------
    # Run Entire Pipeline
    # -------------------------------------------------------

    def run(self):

        self.load_data()

        self.prepare_data()

        self.fit_model()

        self.evaluate_model()

        self.plot_density_heatmap()

        self.plot_training_assignment()

        self.plot_test_assignment()

In [20]:
from google.colab import files

uploaded = files.upload()

Saving CC GENERAL.csv to CC GENERAL (1).csv


In [29]:
segmenter = GMMFinancialSegmenter(
    csv_path="CC GENERAL.csv",
    features=["PURCHASES", "CREDIT_LIMIT"],
    n_components=3
)
segmenter.run()

Dataset Shape : (8949, 2)

Training Samples : 7159
Testing Samples  : 1790

========== EM RESULTS ==========
Converged : True
Iterations : 19

Mixture Weights
[0.45305959 0.44327961 0.10366081]

Means
[[-0.03862204  0.3447703 ]
 [-0.38336541 -0.66634532]
 [ 1.70246139  1.40177383]]

Covariance Matrices

Cluster 1
[[ 0.14734216 -0.09244253]
 [-0.09244253  0.74728805]]

Cluster 2
[[ 0.00925514 -0.00266788]
 [-0.00266788  0.08227737]]

Cluster 3
[[4.51629907 0.20225887]
 [0.20225887 1.79000838]]

========== MODEL PERFORMANCE ==========
Average Train Log-Likelihood : -1.5639
Average Test Log-Likelihood  : -1.6465


ValueError: 
    Invalid value of type 'builtins.str' received for the 'colorscale' property of scatter.marker
        Received value: 'Set1'

    The 'colorscale' property is a colorscale and may be
    specified as:
      - A list of colors that will be spaced evenly to create the colorscale.
        Many predefined colorscale lists are included in the sequential, diverging,
        and cyclical modules in the plotly.colors package.
      - A list of 2-element lists where the first element is the
        normalized color level value (starting at 0 and ending at 1),
        and the second item is a valid color string.
        (e.g. [[0, 'green'], [0.5, 'red'], [1.0, 'rgb(0, 0, 255)']])
      - One of the following named colorscales:
            ['aggrnyl', 'agsunset', 'algae', 'amp', 'armyrose', 'balance',
             'blackbody', 'bluered', 'blues', 'blugrn', 'bluyl', 'brbg',
             'brwnyl', 'bugn', 'bupu', 'burg', 'burgyl', 'cividis', 'curl',
             'darkmint', 'deep',

###4 - Analysis of Results

### Analysis of the Empirical Density Heatmap

The empirical density heatmap provides an overview of the distribution of the training data after feature scaling. Regions with warmer colors correspond to areas containing a higher concentration of observations, while cooler regions represent lower data density. If multiple high-density regions are visible, this suggests that the data possess an underlying multimodal structure, indicating that a Gaussian Mixture Model is appropriate for representing the distribution.

Unlike a single Gaussian distribution, the Gaussian Mixture Model is capable of approximating several modes simultaneously by combining multiple Gaussian components.

---

###Analysis of the Training Assignment Plot

The training assignment plot overlays the observed training data on a continuous contour map derived from the maximum posterior responsibility,

$$
\max_k \gamma_{ik}.
$$

The contour boundaries separate regions where different Gaussian components dominate. Observations located near the center of a cluster generally have one posterior probability close to one, indicating high confidence in their cluster membership.

In contrast, observations located near the boundaries between clusters typically possess similar posterior probabilities for multiple components. These regions represent uncertainty in the assignment and demonstrate the probabilistic nature of Gaussian Mixture Models.

The continuous contour therefore illustrates the soft assignment

$$
E[Z_i \mid X_i=x],
$$

where each location in the feature space is associated with a probability distribution over all clusters rather than a single deterministic label.

---

### Analysis of the Test Assignment Plot

The test assignment plot applies the learned Gaussian Mixture Model to previously unseen observations.

Most test observations are assigned to regions that correspond closely to the learned cluster structure, indicating that the model generalizes well beyond the training data.

Points located close to the decision boundaries exhibit lower confidence because multiple Gaussian components assign comparable probabilities to these observations. Such behaviour is expected in probabilistic clustering, where uncertainty is explicitly represented through posterior probabilities instead of hard assignments.

A relatively high average test log-likelihood further indicates that the learned density functions provide a good representation of the underlying data distribution.

---

### Interpretation of the Soft Assignment

One of the key characteristics of the Gaussian Mixture Model is that it performs **soft clustering** rather than hard clustering.

For every observation,

$$
x_i,
$$

the Expectation step computes the posterior probabilities

$$
\gamma_{ik}
=
P(C_i=k \mid X_i=x_i).
$$

These probabilities form the conditional expectation

$$
E[Z_i \mid X_i=x_i]
=
\begin{bmatrix}
\gamma_{i1}\\
\gamma_{i2}\\
\vdots\\
\gamma_{iK}
\end{bmatrix},
$$

which represents the expected cluster membership vector.

Instead of assigning each observation exclusively to one cluster, the Gaussian Mixture Model allows partial membership across all clusters. The contour map visualizes these posterior probabilities over the entire feature space, illustrating how cluster membership changes smoothly rather than abruptly.

---

### Overall Conclusion

Gaussian Mixture Models provide a probabilistic approach to clustering by repeatedly applying Bayesian conditional updates.

Initially, each Gaussian component is associated with a prior probability represented by the mixture weight,

$$
\phi_k.
$$

For each observation, the Gaussian likelihood evaluates how compatible the observation is with each cluster. Bayes' theorem combines the likelihood and prior to compute the posterior responsibility,

$$
\gamma_{ik},
$$

which represents the updated probability that the observation belongs to cluster $k$.

The Expectation-Maximization algorithm alternates between computing these posterior probabilities (E-step) and updating the model parameters (M-step). During the M-step, the responsibilities act as fractional membership weights for estimating the mixture weights, cluster means, and covariance matrices.

Consequently, Gaussian Mixture Models perform clustering through repeated conditional expectation updates of latent cluster membership variables, producing flexible probabilistic clusters that naturally represent uncertainty in the data.